In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


import sys
sys.path.append("../utils")
from plotting_utils import format_top_3, plot_metric_grouped_by, plot_bio_vs_batch_correction,  plot_bio_vs_batch_correction_markers

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# base_path = ".."
# mali_path = "../../MALI"
# scratch_path = ".."

# save_name = f"real_batches" #dataset}".format(dataset = dataset_name)
# save_path = f"{scratch_path}/results/{save_name}"


# results_path = "../results"
results_path = "../results"
exp_name = "real_batches_lung"
save_name = f"real_batches_lung/2026-05-01_01-25-45" #dataset}".format(dataset = dataset_name)
save_path = f"{results_path}/{save_name}"


create a new summary df based on aggregating the dfs in the n_comp subfolders (only needs to be ran once)

In [ ]:
batches = ["1","2","3","4","5","6", "A1", "A2", "A3", "A4", "A5", "A6", "B1", "B2", "B3", "B4"]
batches = ["A1", "A2", "A3", "A4", "A5", "A6"]
seeds = [11784, 39041, 56089, 79121, 4386721]

all_results_df = pd.DataFrame()
for i1, batch1 in enumerate(batches):
    for i2, batch2 in enumerate(batches):
        if i1 >= i2:
            continue
    
        for seed in seeds:
            save_path_subfolder = f"{save_path}/{batch1}_{batch2}/seed_{seed}"
            try:
                df = pd.read_csv(f"{save_path_subfolder}/{exp_name}_results.csv")
            except FileNotFoundError:
                df = pd.DataFrame()
                print(f"File not found: {save_path_subfolder}/{exp_name}_results.csv")

            df["batch1"] = batch1
            df["batch2"] = batch2
            all_results_df = pd.concat([all_results_df, df], ignore_index=True)
                
all_results_df    
all_results_df.to_csv(f"{save_path}/{exp_name}_with_batches_results.csv", index=False)
                

In [ ]:
save_name

# Real batches data

In [ ]:
# load results
exp_name = "real_batches_lung_with_batches"
results_df = pd.read_csv(f"{save_path}/{exp_name}_results.csv")


metric_type = results_df.loc[results_df["method"] == "Metric Type"]
metric_type = metric_type.drop(columns=['method']).drop_duplicates()
results_df = results_df.loc[results_df["method"] != "Metric Type"]


# # convert relevant columns to numeric
num_cols = results_df.columns.difference(['method', 'batch1', 'batch2'])
results_df[num_cols] = results_df[num_cols].apply(pd.to_numeric, errors='coerce')


# results_df = results_df.drop(0, axis = 0) # remove one run from debugging
# results_df.drop_duplicates(subset=['method'], keep='last', inplace=True)


# delete rows where n_components is NA
results_df = results_df.loc[~results_df["n_components"].isna()]


# remove n_components in the model column (it was just used to not overwrite adata, and it's already another column)
results_df["method"] = results_df["method"].str.replace(r'_\d+_components', '', regex=True)

In [ ]:
# rename FOSTA PHATE t2 to FoSTA
results_df["method"] = results_df["method"].replace({"FoSTA_PHATE_t2": "FoSTA"})
results_df["method"] = results_df["method"].replace({"RFMALI_PHATE_t2": "RFMALI"})

methods_to_remove= ["RFMALI"]
results_df = results_df[~results_df["method"].isin(methods_to_remove)]

In [ ]:
results_df

# make plots

In [ ]:
plot_metric_grouped_by(results_df, groupby_cols=["method"])

In [ ]:
plot_metric_grouped_by(results_df, groupby_cols=["batch1_batch2", "method"], annotate=False, fig_len_factor=4)

In [ ]:
results_df["batch1_batch2"] = results_df["batch1"] + "_" + results_df["batch2"]

In [ ]:
plot_bio_vs_batch_correction(results_df)

In [ ]:
plot_bio_vs_batch_correction_markers(results_df)

In [ ]:
batch1 = "A4"
batch2 = "A5"
results_subset = results_df[(results_df["batch1"] == batch1) & (results_df["batch2"] == batch2)]
plot_bio_vs_batch_correction_markers(results_subset, title="Batch pair: {}-{}".format(batch1, batch2))

In [ ]:
# bar plot of the std over bio conservation and batch correction for each method
std_df = (
    results_df.groupby("method")[["Bio conservation", "Batch correction"]]
    .std()
    .fillna(0)
)

ax = std_df.plot(kind="bar", figsize=(10, 6))
ax.set_ylabel("Standard deviation")
ax.set_title("Std of Bio conservation and Batch correction by method")
plt.tight_layout()
plt.show()

In [ ]:
std_df = results_df.groupby(['method', 'batch1', 'batch2'])[["Bio conservation", "Batch correction"]].std()
std_df = std_df.fillna(0)


In [ ]:
std_df.groupby("method")[["Bio conservation", "Batch correction"]].mean().plot(kind="bar", figsize=(10, 6))
plt.ylabel("Mean standard deviation")
plt.title("Mean std of Bio conservation and Batch correction by method")
plt.tight_layout()

In [ ]:
results_df["batch_pair"] = f"{results_df['batch1']}_{results_df['batch2']}"
results_df["batch_pair"].unique()

In [ ]:
results_df

# only consider batches within the same set

In [ ]:
def get_set_of_batch(batch):
    if batch.startswith("A"):
        return "A"
    elif batch.startswith("B"):
        return "B"
    else:
        return "0"
    
results_df["same_batch_set"] = (results_df["batch1"].apply(get_set_of_batch) == results_df["batch2"].apply(get_set_of_batch))

In [ ]:
same_batch_results_df = results_df.loc[results_df["same_batch_set"] == True]

In [ ]:
plot_bio_vs_batch_correction(same_batch_results_df)